# MEN 4338 · Week 2 — Compositing and declustering

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/iugb/men4338/notebooks/MEN4338_W2_compositing_declustering.ipynb)

**IUGB · BSc Mining Engineering · Geostatistics (lecture & lab)** · Instructor: Laurent Kouadio

The laboratory does not return one grade per hole: it returns one assay per **interval** of core or RC chips,
with different lengths, and sometimes no assay at all (lost or unsampled interval). Before any statistics we
bring the data to a common **support** by compositing. Then we deal with the second trap seen in week 1: the
infill holes are clustered on the rich zone, so the plain average over-estimates the deposit. **Declustering**
gives each sample a weight that reflects the area it represents.

**What you will practise:** `groupby` and length-weighted means · missing values · composite checks ·
cell declustering · choosing the cell size.

Run each cell with **Shift + Enter**, from top to bottom, starting with the two setup cells.

In [ ]:
# Setup: run this cell first (Colab or local Jupyter).
# It fetches the data and the answer checks: from the repository (../data, ../tests)
# if you downloaded it, otherwise from GitHub.
import os, shutil, urllib.request
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/iugb/men4338/'
FILES = [('data/bassam_intervals.csv', 'bassam_intervals.csv'), ('data/bassam_samples.csv', 'bassam_samples.csv'), ('tests/w2/q1.py', 'tests/q1.py'), ('tests/w2/q2.py', 'tests/q2.py'), ('tests/w2/q3.py', 'tests/q3.py'), ('tests/w2/q4.py', 'tests/q4.py'), ('tests/w2/q5.py', 'tests/q5.py')]
os.makedirs('tests', exist_ok=True)
for source, target in FILES:
    if os.path.exists(target):
        continue
    local = os.path.join('..', source)
    if os.path.exists(local):
        shutil.copy(local, target)                      # offline kit / cloned repo
    else:
        urllib.request.urlretrieve(URL + source, target)  # Colab: download
print('Ready:', ', '.join(t for _, t in FILES))

In [ ]:
# Automatic answer checks (Otter-Grader)
try:
    import otter
except ImportError:
    %pip install -q otter-grader
    import otter
grader = otter.Notebook(tests_dir="tests")

## 1. The raw intervals

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

intervals = pd.read_csv("bassam_intervals.csv")
samples = pd.read_csv("bassam_samples.csv")
print(intervals.shape)
intervals.head(8)

Each hole samples the same 10 m bench (`from_m` 0 to `to_m` 10) in intervals of 1, 1.5 or 2 m. An empty
`au_gpt` cell means *not sampled* (NaN). Never replace it by 0: that would invent barren rock.

## 2. Compositing

A **composite** is the length-weighted mean of the intervals over a fixed length (here the whole bench):

$$z_c = \frac{\sum_i z_i \, \ell_i}{\sum_i \ell_i}$$

summed over the **assayed** intervals only. One hole by hand:

In [ ]:
h = intervals[intervals["hole_id"] == "BR-001"].copy()
h["length"] = h["to_m"] - h["from_m"]
print(h)
print("composite:", (h["au_gpt"] * h["length"]).sum() / h["length"].sum())

Why weight by length? A 2 m interval represents twice as much rock as a 1 m interval. A plain average of the
intervals gives the same importance to both and is biased when lengths and grades are related.

## 3. Declustering

With **cell declustering**, the area is divided into square cells; every cell that contains data gets the
same total weight, shared equally between its samples. A sample alone in its cell weighs more than a
sample in a crowded cell:

$$w_i \propto \frac{1}{n_{cell(i)}}, \qquad \sum_i w_i = 1$$

The cell size matters: too small and every sample is alone (no declustering), too large and everything
falls in one cell. A common choice is the spacing of the regular grid.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(samples["x"], samples["y"], s=8, c=samples["campaign"].eq("infill"), cmap="coolwarm")
for v in range(0, 1201, 100): ax.axvline(v, lw=0.3, color="grey")
for v in range(0, 1001, 100): ax.axhline(v, lw=0.3, color="grey")
ax.set_aspect("equal"); ax.set_title("100 m declustering cells")
plt.show()

---
## Exercises

### Q1. Interval lengths and missing assays

Add a column `length_m` to `intervals`. Store in `n_missing` the number of intervals without an assay and in
`sampled_m` the total length (m) of the intervals that do have one.

In [ ]:
intervals["length_m"] = ...
n_missing = ...
sampled_m = ...
n_missing, sampled_m

In [ ]:
grader.check("q1")

### Q2. Composite every hole

Build `composites`, a pandas Series indexed by `hole_id` holding the length-weighted composite of each hole
(ignore the missing intervals). Then check it against `samples`: store in `max_diff` the largest absolute
difference between your composites and the `au_gpt` column of `bassam_samples.csv`.

*Hint:* drop the missing rows with `dropna(subset=["au_gpt"])`, compute `au_gpt * length_m`, then
`groupby("hole_id").sum()`. Use `composites.reindex(samples["hole_id"])` to line them up.

In [ ]:
ok = ...
composites = ...
max_diff = ...
max_diff

In [ ]:
grader.check("q2")

### Q3. Cell declustering weights

Write `cell_weights(x, y, cell_size)` returning a NumPy array of weights, one per sample, that sum to 1.
Cells start at the origin: the cell of a sample is `(floor(x / cell_size), floor(y / cell_size))`.

*Hint:* build one key per sample, e.g. `ix * 100000 + iy`, then `np.unique(key, return_inverse=True, return_counts=True)`.

In [ ]:
def cell_weights(x, y, cell_size):
    ...

cell_weights(np.array([10., 20., 30., 160.]), np.array([10., 15., 20., 40.]), 100)

In [ ]:
grader.check("q3")

### Q4. Declustered mean

Store the plain mean of `samples["au_gpt"]` in `naive_mean` and the declustered mean with 50 m cells
(the grid spacing) in `declus_mean`. By how much did the clustering bias the mean?

In [ ]:
naive_mean = ...
declus_mean = ...
naive_mean, declus_mean

In [ ]:
grader.check("q4")

### Q5. Which cell size?

Compute the declustered mean for `cell_sizes = np.arange(25, 201, 25)` and store the results in a list or
array `declus_means`. Plot them against the cell size. Since the clusters sit on high grades, keep the cell
size that gives the **lowest** mean in `best_cell`. (Cells larger than about four times the grid spacing
start to mix unrelated areas, which is why we stop at 200 m.)

In [ ]:
cell_sizes = ...
declus_means = ...
best_cell = ...

plt.plot(cell_sizes, declus_means, "o-")
plt.axhline(naive_mean, ls="--", color="grey")
plt.xlabel("cell size (m)"); plt.ylabel("declustered mean (g/t)")
plt.show()
best_cell

In [ ]:
grader.check("q5")

### Going further (not checked)

* Draw the histogram of the grades with and without the 50 m weights (`plt.hist(au, weights=w)`).
* Polygonal declustering: weight each sample by the area of its nearest-neighbour polygon (approximate it
  by counting, for each sample, the 10 m cells of the bench closest to it).

---
**Before you leave:** run all the checks with the cell below, then save your notebook
(**File → Download → .ipynb** in Colab).

In [ ]:
grader.check_all()